# Kuroshio Core：所有现有模型的区域训练实验

这个 notebook 回答一个非常具体的问题：**如果只使用 Kuroshio Core 的 training cases 重新训练模型，再预测固定的 Kuroshio test cases，72 小时 RMSE 是否会大幅下降？**

实验保持原来的 drifter-level training/calibration/test 划分不变，因此同一个 drifter 不会跨集合。区域筛选发生在原始划分之后。

## 数据和公平比较

- Kuroshio training：646 cases，208 drifters。
- Kuroshio test：139 cases，45 drifters。
- Calibration partition 未用于 RQ1 模型选择。
- Ridge、FNN 和 Weighted KNN 的超参数只在 Kuroshio training data 内，通过按 drifter 分组的 5-fold CV 选择。
- VAR(1) 只使用这些 Kuroshio training cases 的 24 小时历史窗口内的一小时速度转移。
- Persistence 没有需要训练的参数；poster 中的 Mean Flow 是全区域 training data 构造的基线，所以这两项只作为参照，不冒充区域重训结果。

In [ ]:
from pathlib import Path
import pandas as pd
from IPython.display import display, Image

ROOT = next(
    p for p in [Path.cwd(), *Path.cwd().parents]
    if (p / 'RQ2' / 'data' / 'kuroshio_all_models').exists()
)
OUT = ROOT / 'RQ2' / 'data' / 'kuroshio_all_models'
FIG = ROOT / 'RQ2' / 'figures' / 'kuroshio_all_models'
results = pd.read_csv(OUT / 'kuroshio_all_six_models_summary.csv')
display(results.round(2))

## 核心结果

|模型|全区域 RMSE|区域 RMSE|RMSE 改善|全区域 Median|区域 Median|Median 改善|
|---|---:|---:|---:|---:|---:|---:|
|FNN|87.48 km|**82.53 km**|5.66%|64.08 km|**62.03 km**|3.21%|
|Ridge|92.08 km|86.26 km|6.32%|66.66 km|66.05 km|0.92%|
|Weighted KNN|92.66 km|90.26 km|2.59%|67.44 km|67.35 km|0.14%|
|VAR(1)|111.47 km|94.34 km|15.37%|91.38 km|78.04 km|14.60%|
|Velocity persistence|120.36 km|不适用|—|83.32 km|不适用|—|
|Spatial mean flow|105.33 km|未区域重训|—|77.10 km|未区域重训|—|

结论：**四个学习模型的 RMSE 都有所下降，但最好的 Kuroshio-only FNN 仍有约 82.5 km 的 72-h RMSE。** 因此，Kuroshio 的高误差不是简单地把其他海区训练样本删掉就能解决的。

In [ ]:
display(Image(filename=str(FIG / 'kuroshio_global_vs_regional_all_models.png')))

## RMSE 和 Median error 应该怎样看？

对于每一个 test case，我们先计算预测终点与真实终点之间的 Haversine separation error。72-h RMSE 和 median error 都来自这 139 个 case 的终点误差，但汇总方式不同。

**Median error** 是排序后位于中间的误差，表示一个“典型 test case”大约会错多远。它不容易被少数特别差的预测拉高。

**RMSE** 先平方每个误差、求平均再开根号。由于平方会放大大误差，它对少数严重失败案例特别敏感。RMSE 明显高于 median，意味着误差分布存在较长的右尾或一些极端失败案例。

因此需要把两者一起看：

1. **RMSE 和 median 都下降**：典型案例和极端案例整体都在改善。VAR(1)属于这种情况，但它最终仍不是最好的模型。
2. **RMSE 明显下降、median 几乎不变**：改善主要来自减少少数大误差，而大多数典型案例变化有限。Ridge 和 KNN 更接近这种情况。
3. **比较最终模型优劣时看绝对值，而不仅看改善百分比。** VAR(1)改善比例最大，但区域 RMSE 仍为94.34 km；FNN的改善比例较小，但绝对 RMSE 最低，为82.53 km。
4. **绝对误差仍然很大。** 只做区域化并不能提供未来72小时随时间变化的洋流、风和涡旋信息。
5. **不能据此证明‘流速快导致误差大’。** 目前只能说明 Kuroshio 同时具有较快流速、较长位移和较高预测误差，因果解释还需要控制位移长度、季节等因素。

## 超参数选择结果

- Ridge：alpha = 10。
- Weighted KNN：K = 20，distance power p = 2。
- FNN：128 → 64 两层隐藏层，L2 = 0.001，最终训练 94 epochs。
- VAR(1)：没有需要搜索的超参数；使用 14,858 个去重的一小时速度转移。

## 如需从头重跑

在项目根目录的 Terminal 运行：

```bash
cd "/Users/roger/Documents/data3001-project"
source .venv/bin/activate
export MPLCONFIGDIR=/tmp/data3001-matplotlib-all-models
python RQ2/19_kuroshio_all_models.py
```

脚本会覆盖本 notebook 使用的结果表和图片。